# CVM Training — SageMaker Edition

Court Vision Mapping (33-keypoint pose model) training on Amazon SageMaker.

**Why this notebook differs from the Colab version:**
- Writes runs to **persistent EBS** (`/home/ec2-user/SageMaker/` on Notebook Instances, `/root/` on Studio) instead of an ephemeral VM — survives kernel/runtime restarts.
- Optional **S3 sync** during training so checkpoints back up even if the instance dies.
- No `google.colab` calls; data is pulled from S3 (or already on EBS).
- Bigger `batch` / `workers` defaults tuned for ml.g5.xlarge (A10G, 24 GB VRAM, 4 vCPU).

**Recommended instance:** `ml.g5.xlarge` (≈$1.41/hr on-demand) — A10G is ~2× faster than the Colab T4 you had.

If you can't get g5 quota, `ml.g4dn.xlarge` (T4) is the fallback.

## 0. Environment check
Confirm GPU, CUDA, and where to put persistent outputs.

In [ ]:
import os, subprocess, torch, platform
from pathlib import Path

print(f"Python      : {platform.python_version()}")
print(f"Torch       : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# SageMaker Notebook Instances → /home/ec2-user/SageMaker (persistent EBS)
# SageMaker Studio          → /root/ or /home/sagemaker-user/ (persistent EBS volume)
for candidate in ["/home/ec2-user/SageMaker", "/home/sagemaker-user", "/root"]:
    if Path(candidate).exists():
        WORKSPACE = Path(candidate)
        break
else:
    WORKSPACE = Path.cwd()

PROJECT_DIR = WORKSPACE / "basketball-cvm"
PROJECT_DIR.mkdir(exist_ok=True)
os.chdir(PROJECT_DIR)
print(f"\nWorkspace   : {WORKSPACE}")
print(f"Project dir : {PROJECT_DIR}")

## 1. Install dependencies
SageMaker kernels usually have torch + CUDA pre-installed. Only install what's missing.

In [ ]:
%pip install -q ultralytics supervision pyyaml

## 2. Pull data from S3

Upload your dataset zips to S3 once (e.g. via `aws s3 cp` from your local machine), then this cell mirrors them to the instance's local SSD where I/O is fastest. Training off S3 directly is much slower than EBS.

**Set these two variables:**

In [ ]:
S3_DATA_URI = "s3://YOUR-BUCKET/basketball/data/"   # folder containing the dataset zips
S3_RUNS_URI = "s3://YOUR-BUCKET/basketball/runs/"   # where to back up training outputs

DATA_DIR = PROJECT_DIR / "Data"
DATA_DIR.mkdir(exist_ok=True)

# Sync zips from S3 → local EBS (skips files already present)
!aws s3 sync {S3_DATA_URI} {DATA_DIR} --exclude "*" --include "*.zip"
!ls -lh {DATA_DIR}

In [ ]:
# Unzip all datasets into Data/<name>/
import zipfile, glob

for path in glob.glob(str(DATA_DIR / "*.zip")):
    name = os.path.splitext(os.path.basename(path))[0]
    out = DATA_DIR / name
    if out.exists() and any(out.iterdir()):
        print(f"Skip (already extracted) → {out}")
        continue
    print(f"Extracting → {out}")
    with zipfile.ZipFile(path) as z:
        z.extractall(out)

## 3. Build the dataset YAML
33-keypoint pose dataset. Rewrites paths to absolute so Ultralytics resolves them regardless of CWD.

In [ ]:
import yaml

CVM_root = (DATA_DIR / "basketball-court-detection-2.v1-1.yolov8").resolve()
src_yaml = CVM_root / "data.yaml"

with open(src_yaml) as f:
    cfg = yaml.safe_load(f)

cfg["path"]  = str(CVM_root)
cfg["train"] = str(CVM_root / "train" / "images")
cfg["val"]   = str(CVM_root / "valid" / "images")
cfg["test"]  = str(CVM_root / "test"  / "images")

data_yaml_path = (PROJECT_DIR / "cvm_data.yaml").resolve()
with open(data_yaml_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"Wrote {data_yaml_path}")
print(f"  nc={cfg['nc']} names={cfg['names']} kpt_shape={cfg.get('kpt_shape')}")

## 4. Train

Key knobs (vs the Colab run that converged at epoch 367):
- `epochs=400` — known to be enough for this dataset
- `patience=30` — early-stop on `fitness` (weighted box+pose mAP50-95)
- `batch=32` — A10G can handle this comfortably at imgsz=640; lower to 16 if you're on T4
- `workers=4` — ml.g5.xlarge has 4 vCPU
- `project=` points at persistent EBS, so `best.pt` survives instance restart
- `save_period=25` — keep a periodic checkpoint for S3 backup

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n-pose.pt")   # auto-downloads on first use
model.info()

RUNS_DIR = PROJECT_DIR / "runs"

results = model.train(
    data=str(data_yaml_path),
    epochs=400,
    patience=30,
    imgsz=640,
    batch=32,                  # A10G/24GB. Drop to 16 on T4/16GB.
    workers=4,                 # match instance vCPU count
    device=0,
    project=str(RUNS_DIR),
    name="CVM",
    exist_ok=True,
    deterministic=True,
    plots=True,
    save_period=25,            # checkpoint every 25 epochs for S3 backup
    amp=True,                  # mixed precision on GPU
)

## 5. Back up to S3
Sync the run directory (weights + plots + logs) to S3. Run this **immediately after training finishes**, and feel free to re-run anytime.

In [ ]:
!aws s3 sync {RUNS_DIR} {S3_RUNS_URI} --exclude "*.pt.bak"
print(f"Synced → {S3_RUNS_URI}")

### Optional: background S3 sync during training
If you want belt-and-braces protection (and you should — that's how the Colab run got lost), open a **separate terminal** in SageMaker and run:

```bash
while true; do
  aws s3 sync /home/ec2-user/SageMaker/basketball-cvm/runs s3://YOUR-BUCKET/basketball/runs/
  sleep 300
done
```

That mirrors checkpoints to S3 every 5 minutes regardless of what the kernel is doing.

## 6. Sanity check — single-frame keypoint detection

In [ ]:
import supervision as sv
import numpy as np

best_pt = RUNS_DIR / "pose" / "CVM" / "weights" / "best.pt"
if not best_pt.exists():
    # Older Ultralytics nests as runs/CVM/weights/best.pt
    best_pt = next(RUNS_DIR.rglob("best.pt"))
print(f"Using weights: {best_pt}")

CVM_model = YOLO(str(best_pt))

SOURCE_VIDEO_PATH = str(DATA_DIR / "boston-celtics-new-york-knicks-game-1-q1-01.54-01.48.mp4")
frame = next(sv.get_video_frames_generator(SOURCE_VIDEO_PATH))

result = CVM_model.predict(frame, conf=0.3, verbose=False)[0]
kps = sv.KeyPoints.from_ultralytics(result)

# Zero out low-confidence vertices so the annotator skips them
mask = kps.confidence > 0.5
filtered_xy = np.where(mask[..., None], kps.xy, 0)
kps = sv.KeyPoints(xy=filtered_xy, confidence=kps.confidence)

annotated = sv.VertexAnnotator(color=sv.Color.RED, radius=8).annotate(
    scene=frame.copy(), key_points=kps,
)
sv.plot_image(annotated)

## 7. Shut down the instance
**Don't forget.** A `ml.g5.xlarge` left idle overnight is ~$10. Stop the notebook instance from the SageMaker console when you're done.